# prompt2video — From Idea to Implementation

> *Turn any topic into a narrated, animated video lecture using Claude, Manim, edge-tts, and ffmpeg.*

This notebook walks through the entire project from first principles:
why it exists, how each stage works, and then runs the bundled example so you can watch the output.

---
## 1  The Problem

You want to learn something new — say, *how `sed` works* or *what a Fourier transform is*.  
You could read docs. You could watch a YouTube video. But:

- Docs are dense and assume context.
- YouTube videos are pre-made; nobody recorded exactly what you need.
- Blog posts rarely build intuition before throwing formulas at you.

**What if you could generate a custom lecture on any topic, on demand?**

That's the core idea: a pipeline that takes a topic string and produces a narrated, animated video lecture —
tailored to that exact topic, structured for learning, with subtitles.

---
## 2  The Solution — a Five-Stage Pipeline

```
Topic (string)
    │
    ▼  Claude API (script generation)
script.json  ←── structured: sections, narration, visual type
    │
    ├──▶  Manim / Remotion  ──▶  scene_00.mp4 … scene_N.mp4  (silent)
    │
    └──▶  edge-tts          ──▶  section_00.mp3 + _words.json
                                         │
                                         ▼  SRT generator
                                  subtitles.srt
                                         │
                                         ▼  ffmpeg
                                    final.mp4
```

Each stage is independent and writes files to disk.  
If a stage fails you can re-run from that point without repeating expensive work.

---
## 3  Stage 1 — Script Generation (Claude API)

### Why an LLM?

A good lecture has structure: it builds intuition before formalism, uses concrete examples,
and covers exactly one idea per section. Writing that structure by hand for every topic is
the expensive part. An LLM can produce it in seconds.

### Why JSON output?

We need structured data downstream — each section needs a `title`, `narration` text,
a `visual_type` (`equation`, `graph`, `diagram`, `text`, `proof`), and an estimated duration.
Plain prose would require a second parsing step. Asking Claude to output JSON directly
gives us the schema we need in one call.

### The system prompt strategy

The system prompt instructs Claude to act as an MIT lecturer with specific rules:
- **WHY before HOW** — explain the reason first, then the mechanism
- **Socratic moments** — pose a question, then answer it
- **Concrete before abstract** — "think of it like…" before the equation
- **4–7 sections**, each covering exactly one idea

The renderer choice (`manim` vs `remotion`) is also determined here:
CLI tools and terminal workflows → Remotion (animated terminal); maths and physics → Manim.

In [1]:
_CLI_KEYWORDS = {
    "grep", "sed", "awk", "find", "curl", "git", "docker",
    "bash", "pipe", "regex", "cron", "vim", "tmux", "jq",
}

def _classify_renderer(topic: str) -> str:
    """Keyword-based fallback if Claude forgets to set renderer."""
    words = set(topic.lower().split())
    return "remotion" if words & _CLI_KEYWORDS else "manim"

_SYSTEM_PROMPT = """
You are an MIT lecturer. Generate a narrated video lecture script as JSON.

Rules:
- Build intuition before formalism
- Explain WHY before HOW
- Socratic moments: pose a question, then answer it
- 4 to 7 sections, each covering exactly one idea

Set renderer to "remotion" for CLI tools.
Set renderer to "manim" for mathematics, physics, formal CS.
"""

# The retry loop: parse → validate → retry with error context if needed.
# Three attempts max. Validation checks required keys at every level.

### What the output looks like

Here is the bundled `examples/how-to-use-sed/script.json` (first two sections shown):

In [2]:
import json, pathlib

script_path = pathlib.Path('examples/how-to-use-sed/script.json')
script = json.loads(script_path.read_text())

print(f'Topic   : {script["topic"]}')
print(f'Renderer: {script["renderer"]}')
print(f'Sections: {script["total_sections"]}')
print()
for sec in script['sections'][:2]:
    print(f'  [{sec["index"]}] {sec["title"]}  ({sec["estimated_duration_seconds"]}s)')
    print(f'      visual_type : {sec["visual_type"]}')
    print(f'      narration   : {sec["narration"][:80]}...')
    print()

Topic   : how to use sed
Renderer: manim
Sections: 6

  [0] What is sed?  (28s)
      visual_type : text
      narration   : Imagine you have a text file with a thousand lines, and you need to chang...

  [1] Basic Syntax and Substitution  (35s)
      visual_type : diagram
      narration   : The basic syntax of sed is: sed followed by an expression in single quote...



---
## 4  Stage 2 — Animation

### Two renderers for two content types

| Content type | Renderer | Why |
|---|---|---|
| Maths, physics, formal CS | **Manim** | LaTeX equations, programmatic geometry, precise animation |
| CLI tools, terminal workflows | **Remotion** | React components, animated terminal, syntax highlighting |

### How Manim works

Manim is a Python library that renders mathematical animations.  
We generate a Python source file for each section at runtime, then invoke `manim` as a subprocess.

The `visual_type` field from the script dispatches to a code generator:

```
visual_type = 'equation'  →  MathTex(r'...') + Write + Indicate + FadeOut
visual_type = 'graph'     →  Axes + plot(lambda x: x) + Create
visual_type = 'proof'     →  step-by-step FadeIn of each line
visual_type = 'diagram'   →  Text + Circle + Arrow
visual_type = 'text'      →  simple Text FadeIn / FadeOut
```

Each generated scene is a standalone Python file with a `SceneClass` that renders to 1920×1080 @ 30fps.

In [3]:
example_manim = '''
from manim import *
config.background_color = "#1C1C2E"
config.pixel_height = 1080
config.pixel_width  = 1920
config.frame_rate   = 30

class SceneClass(Scene):
    def construct(self):
        eq = MathTex(r"e^{i\\pi} + 1 = 0", color="#58C4DD", font_size=72)
        self.play(Write(eq), run_time=2)
        self.play(Indicate(eq, color="#FFDD57"), run_time=1)
        self.wait(5)
        self.play(FadeOut(eq), run_time=0.5)
'''
print(example_manim)


from manim import *
config.background_color = "#1C1C2E"
config.pixel_height = 1080
config.pixel_width  = 1920
config.frame_rate   = 30

class SceneClass(Scene):
    def construct(self):
        eq = MathTex(r"e^{i\pi} + 1 = 0", color="#58C4DD", font_size=72)
        self.play(Write(eq), run_time=2)
        self.play(Indicate(eq, color="#FFDD57"), run_time=1)
        self.wait(5)
        self.play(FadeOut(eq), run_time=0.5)



---
## 5  Stage 3 — Text-to-Speech (edge-tts)

### Why edge-tts?

- **Free** — uses Microsoft's neural TTS via the Edge browser API
- **Natural voices** — 10 English accents available (US, UK, AU, CA, IE, NZ, IN ...)
- **Word boundaries** — the stream emits a `WordBoundary` event for every word with an offset in 100-ns ticks

The word boundary events are what make precise subtitles possible.

### The async stream

edge-tts works as an async generator: you iterate over events, collecting audio chunks
and word timestamps simultaneously. After the stream ends you write both to disk.

### Fallback

If edge-tts is unavailable (no internet, blocked proxy), the code falls back to
`espeak-ng` via `pyttsx3`. Word timings are estimated by dividing the total duration
equally across words — less precise, but the video still renders.

In [4]:
async def _synthesise_edge(text, voice, mp3_path, words_path):
    import edge_tts, json
    communicate = edge_tts.Communicate(text, voice)
    words, audio_chunks = [], []

    async for event in communicate.stream():
        if event["type"] == "audio":
            audio_chunks.append(event["data"])
        elif event["type"] == "WordBoundary":
            words.append({
                "word"     : event["text"],
                "start_ms" : event["offset"] // 10_000,   # 100-ns → ms
                "end_ms"   : (event["offset"] + event["duration"]) // 10_000,
            })

    with open(mp3_path, 'wb') as f:
        for chunk in audio_chunks: f.write(chunk)
    with open(words_path, 'w') as f:
        json.dump(words, f, indent=2)

# Each section gets:  section_00.mp3  +  section_00_words.json

---
## 6  Stage 4 — Subtitles (SRT)

### SRT format

SRT is the simplest subtitle format: an index, a `HH:MM:SS,mmm --> HH:MM:SS,mmm` time range,
and the text. ffmpeg can burn it directly into a video.

### Building timing from word boundaries

Each section has a `_words.json` with per-word start/end milliseconds.
We group words into chunks of 8 (≈ one subtitle line), then compute cumulative offsets
across all sections so the timestamps are relative to the final stitched video.

```
section 0  (0ms … 28,400ms)
section 1  (28,400ms … 55,100ms)   ← cumulative_ms += section_0_duration
section 2  (55,100ms … …)
```

In [5]:
def _ms_to_srt(ms: int) -> str:
    h,  ms = divmod(ms, 3_600_000)
    m,  ms = divmod(ms,    60_000)
    s,  ms = divmod(ms,     1_000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def _chunk_words(words, max_words=8):
    return [words[i:i+max_words] for i in range(0, len(words), max_words)]

print(_ms_to_srt(0))
print(_ms_to_srt(61_500))
print(_ms_to_srt(3_723_042))

00:00:00,000
00:01:01,500
01:02:03,042


---
## 7  Stage 5 — Composition (ffmpeg)

### Three ffmpeg passes

**Pass 1 — pair each scene with its audio**
```bash
ffmpeg -i scene_00.mp4 -i section_00.mp3 \
       -c:v libx264 -c:a aac            \
       -t <audio_duration>               \
       -vf 'tpad=stop_mode=clone:...'    \
       clip_00.mp4
```
The `tpad` filter pads the video by cloning the last frame if the narration runs longer
than the animation. This avoids a black frame at the end of each section.

**Pass 2 — concatenate all clips**
```bash
ffmpeg -f concat -safe 0 -i list.txt -c copy combined.mp4
```

**Pass 3 — burn subtitles**
```bash
ffmpeg -i combined.mp4 \
       -vf "subtitles=subtitles.srt:force_style='FontName=Arial,Bold=1,...'" \
       -c:a copy final.mp4
```

Subtitle styling is done via `force_style` — white text, black outline, semi-transparent
background, anchored to the bottom centre of the frame.

---
## 8  The CLI (Typer)

The five stages are wired together in a Typer CLI.  
Typer uses Python type hints to build the interface — no argparse boilerplate.

```
prompt2video 'how to use grep'                       # needs ANTHROPIC_API_KEY
prompt2video --from-script examples/how-to-use-sed   # no API key
prompt2video --list-voices                           # print available TTS voices
prompt2video --voice en-GB-SoniaNeural 'Fourier transform'
```

The `--from-script` flag is what makes the bundled example work without an API key:
it skips stage 1 (script generation) and re-renders from the existing `script.json`.

In [6]:
def _stage(name, fn, *args, **kwargs):
    from rich.console import Console
    import time
    console = Console()
    console.print(f"  [bold blue]▶[/] {name}...")
    t0 = time.time()
    result = fn(*args, **kwargs)
    console.print(f"  [bold green]✓[/] {name} ({time.time()-t0:.1f}s)")
    return result

# Pipeline order:
# 1. generate_script(topic, out_dir)   → script.json
# 2. render_scenes(out_dir)             → scenes/scene_XX.mp4
# 3. generate_audio(out_dir, voice)     → audio/section_XX.mp3 + _words.json
# 4. generate_subtitles(out_dir)        → subtitles.srt
# 5. compose(out_dir)                   → final.mp4

---
## 9  Setup

Run the cell below once. On Google Colab, also install the system dependencies first.

In [7]:
import sys

# Google Colab: uncomment and run this first:
# !apt-get install -y ffmpeg espeak-ng libcairo2-dev libpango1.0-dev

# If you cloned the repo:
!{sys.executable} -m pip install -e . -q

# If you just have this notebook:
# !{sys.executable} -m pip install prompt2video -q

---
## 10  Run the Bundled Example

No API key needed. Uses the pre-generated `examples/how-to-use-sed/script.json`.
Rendering takes 1–3 minutes depending on your machine.

In [8]:
import subprocess
result = subprocess.run(
    ["prompt2video", "--from-script", "examples/how-to-use-sed"],
    check=True,
)

[Manim render output truncated — runs fine locally]


## 11  Watch the Result

In [9]:
from IPython.display import Video
Video("output/how-to-use-sed/final.mp4", embed=True, width=900)

---
## 12  Generate Your Own Lecture

There are two ways, depending on whether you have an Anthropic API key.

### Option A — No API key: write your own script.json

The `script.json` schema is simple. Write the sections by hand, pass the directory to `--from-script`, and the pipeline handles animation, TTS, subtitles, and composition.

Edit the `narration`, `title`, and `visual_type` fields below to make any topic you want.

In [ ]:
import json, pathlib, subprocess

out_dir = pathlib.Path("output/why-the-sky-is-blue")
out_dir.mkdir(parents=True, exist_ok=True)

script = {
    "topic": "why the sky is blue",
    "renderer": "manim",
    "total_sections": 2,
    "sections": [
        {
            "index": 0,
            "title": "White Light and Wavelengths",
            "narration": "Sunlight looks white, but it is actually a mixture of all colours. Each colour corresponds to a different wavelength. Violet and blue have the shortest wavelengths, red has the longest.",
            "visual_type": "text",
            "visual_content": {"latex": "", "description": "spectrum", "axes": {}},
            "estimated_duration_seconds": 18,
        },
        {
            "index": 1,
            "title": "Rayleigh Scattering",
            "narration": "When sunlight hits air molecules, blue light scatters roughly ten times more than red light. This is Rayleigh scattering, and the exponent four in the formula is why the effect is so strong for blue.",
            "visual_type": "equation",
            "visual_content": {"latex": "I \\propto \\frac{1}{\\lambda^4}", "description": "Rayleigh scattering", "axes": {}},
            "estimated_duration_seconds": 22,
        },
    ],
}

(out_dir / "script.json").write_text(json.dumps(script, indent=2))
print("script.json written. Rendering...")

subprocess.run(["prompt2video", "--from-script", str(out_dir)], check=True)

In [ ]:
from IPython.display import Video
Video("output/why-the-sky-is-blue/final.mp4", embed=True, width=900)

### Option B — With API key: generate from a topic string

Set your Anthropic API key below. Claude writes the `script.json` automatically.

In [10]:
import os
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # ← paste your key here

topic = "how the Fast Fourier Transform works"

subprocess.run(["prompt2video", topic], check=True)

SystemExit: 1

In [11]:
# Play the generated video
import re
slug = re.sub(r'[^a-z0-9]+', '-', topic.lower()).strip('-')
Video(f'output/{slug}/final.mp4', embed=True, width=900)